# 05 — Apply reviewed comments through PLEB

**You will:** read the saved review, check its hashes, write an apply-only config,
create the `reviewed` branch and audit every active/physical count change.
Do not rerun detection to manufacture a different explanation for a previous decision.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, tarfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tomlkit
COURSE = Path(os.environ['PLEB_COURSE_HOME'])
sys.path.insert(0, str(COURSE))
import release_tools as lab
ROOT = lab.environment(Path(os.environ['PLEB_LEARNING_OUTPUT']))
REPO = ROOT/'dataset-repo'
CONFIGS = ROOT/'configs'
PULSARS = lab.PULSARS
CLI = [sys.executable, '-m', 'pleb.cli']

In [ ]:
approval = json.loads((ROOT/'review_approval.json').read_text())
lab.verify_hashes(ROOT/'review',approval['files'])
common = json.loads((CONFIGS/'common.json').read_text())
apply = dict(common, branches=['reviewed'],reference_branch='reviewed',outdir_name='05_apply',
    run_fix_dataset=True,fix_apply=True,fix_base_branch='prepared',fix_branch_name='reviewed',
    fix_commit_message='Apply reviewed training QC comments',fix_qc_apply_only=True,
    fix_qc_results_dir=str(ROOT/'review'),fix_qc_branch='.',fix_qc_require_csv=True,
    fix_qc_remove_outliers=True,fix_qc_outlier_cols=['reviewed_bad_point'],
    fix_qc_review_include_actions=['mark_bad'],
    fix_qc_action='comment',fix_qc_remove_bad=True,fix_qc_remove_transients=False,
    fix_qc_remove_solar=False,fix_qc_remove_orbital_phase=False,
    fix_qc_write_pqc_flag=False,fix_qc_write_explicit_flags=False,
    fix_qc_write_metric_flags=False,fix_qc_merge_tol_days=0.01/86400)
path = lab.write_config(CONFIGS/'05_apply.toml',apply)
lab.command(CLI+['--config',path],log=ROOT/'logs/05_apply.log')
reviewed_data = lab.snapshot(REPO,'reviewed',ROOT/'exports/05_reviewed')
before = lab.inventory(ROOT/'exports/02_prepared/data')
after = lab.inventory(reviewed_data)
display(after)
expected = len(approval['selected_bad'])
assert after.physical.sum()==before.physical.sum()==580
assert before.active.sum()-after.active.sum()==expected
assert after.commented.sum()==4+expected
assert after.extra_active_duplicates.sum()==0
lab.command(['git','-C',REPO,'diff','--stat','prepared','reviewed'])

`fix_qc_results_dir` identifies the reviewed tables, while `fix_qc_branch` selects a
branch-specific subdirectory. Here `.` means the files are directly under `review/`.
We choose an explicit 0.01-second match tolerance for this synthetic cadence. Do not
enlarge it to hide unmatched rows: close-in-time observations can be different measurements.

Decisions remain in sidecar CSVs with reasons and hashes. TIM comments give the active
selection without exhausting Tempo2's compiled flag limits. Counts are necessary but
insufficient: inspect which rows changed and check no retained measurement was displaced.

`fix_qc_review_include_actions=['mark_bad']` restricts this run to explicit rejections.
It does not apply every automatic `bad_point` or mark every unreviewed row as approved.
A `keep` decision on a previously commented row is a separate, deliberate restoration:
apply only approved keep actions in a separate config and audit the exact reactivated rows.
Do not mix that operation into this rejection-only stage.

In [ ]:
from pleb.release_bundle import dataset_rows, match_rows
old = dataset_rows(ROOT/'exports/02_prepared/data')
new = dataset_rows(reviewed_data)
changed = []
for name in old:
    pairs, absent_old, absent_new = match_rows(old[name],new[name],tolerance=lab.Decimal('0'))
    assert not absent_old and not absent_new, 'A physical observation was added, lost or shifted'
    for i,j in pairs.items():
        a,b = old[name][i],new[name][j]
        assert not (a.commented and not b.commented), 'An earlier exclusion was unexpectedly restored'
        if a.commented!=b.commented:
            changed.append(dict(tim=name,mjd=str(a.mjd),old=a.raw,new=b.raw))
display(pd.DataFrame(changed))
assert len(changed)==expected

**Your task:** reconcile the approved IDs, exact changed rows, Git diff and counts.
Explain why an unmatched or unexpectedly changed row must stop a release even if the
global count difference is correct. Next: fit the final retained selection.